# 04 Advanced Statistical & Risk Analysis

**Problem Statement:** Financial institutions face the challenge of accurately assessing creditworthiness while understanding modern consumer behavior. This project aims to analyze customer spending patterns to gain insights into financial stability and integrate these findings with loan risk assessment models. By leveraging data-driven extraction and statistical analysis, we seek to predict potential default risks and provide actionable insights for optimized lending strategies.

## 1. Advanced Data Extraction & Feature Engineering
We extract new fields like Age, Transaction Frequency, Deposit/Withdrawal behavior, and Loan-to-Balance Ratios to accurately assess creditworthiness.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime

# Ensure output directories exist
os.makedirs('../tables', exist_ok=True)
os.makedirs('../graphs', exist_ok=True)

# Load datasets
dim_customers = pd.read_csv('../data/processed/dim_customers_accounts.csv')
fact_tx = pd.read_csv('../data/processed/fact_transactions.csv')
fact_loans = pd.read_csv('../data/processed/fact_loans.csv')

# Extract Age
dim_customers['DateOfBirth'] = pd.to_datetime(dim_customers['DateOfBirth'], errors='coerce')
dim_customers['Age'] = (datetime.now() - dim_customers['DateOfBirth']).dt.days / 365.25
dim_customers['Age_Group'] = pd.cut(dim_customers['Age'], bins=[18, 25, 35, 50, 100], labels=['18-25', '26-35', '36-50', '50+'])

# Extract Transaction Behavior
tx_summary = fact_tx.groupby(['AccountOriginID', 'TypeName']).agg(Total_Amount=('Amount', 'sum'), Tx_Count=('TransactionID', 'count')).reset_index()
deposits = tx_summary[tx_summary['TypeName'] == 'Deposit'].rename(columns={'Total_Amount': 'Total_Deposits', 'Tx_Count': 'Deposit_Count', 'AccountOriginID':'AccountID'}).drop(columns=['TypeName'])
withdrawals = tx_summary[tx_summary['TypeName'] == 'Withdrawal'].rename(columns={'Total_Amount': 'Total_Withdrawals', 'Tx_Count': 'Withdrawal_Count', 'AccountOriginID':'AccountID'}).drop(columns=['TypeName'])

tx_behavior = fact_tx.groupby('AccountOriginID').agg(Total_Tx_Count=('TransactionID', 'count')).reset_index().rename(columns={'AccountOriginID':'AccountID'})
tx_behavior = tx_behavior.merge(deposits, on='AccountID', how='left').merge(withdrawals, on='AccountID', how='left').fillna(0)

# Merge everything into a Risk Dataframe
fact_loans.rename(columns={'StatusName': 'LoanStatusName'}, inplace=True)
dim_customers.rename(columns={'StatusName': 'AccountStatusName'}, inplace=True)

risk_df = fact_loans.merge(dim_customers, on='AccountID', how='left')
risk_df = risk_df.merge(tx_behavior, on='AccountID', how='left')

# Extract Risk Metrics
risk_df['Loan_to_Balance_Ratio'] = risk_df['PrincipalAmount'] / (risk_df['Balance'] + 1)
risk_df['Is_Overdue'] = np.where(risk_df['LoanStatusName'] == 'Overdue', 1, 0)

display(risk_df.head())

,LoanID,AccountID,LoanStatusID,PrincipalAmount,InterestRate,StartDate,EstimatedEndDate,LoanStatusName,CustomerID,AccountTypeID,...,Country,Age,Age_Group,Total_Tx_Count,Total_Deposits,Deposit_Count,Total_Withdrawals,Withdrawal_Count,Loan_to_Balance_Ratio,Is_Overdue
0,400230,200876,1,76958.56,0.0547,2022-11-20 00:00:00.000000,2026-08-06 00:00:00.000000,Active,10230,1,...,United States,58.633812,50+,37,33962.05,11,38236.90,14,2.347066,0
1,400307,200789,1,29013.67,0.0321,2022-02-22 00:00:00.000000,2025-12-08 00:00:00.000000,Active,10579,5,...,United States,56.106776,50+,30,29662.34,11,10982.87,7,0.743312,0
2,400233,201275,1,48596.76,0.1017,2021-11-21 00:00:00.000000,2023-07-30 00:00:00.000000,Active,10505,5,...,United States,NaN,NaN,26,2601.78,2,22648.31,10,0.874264,0
3,400100,200070,1,9191.43,0.0999,2021-08-14 00:00:00.000000,2023-09-18 00:00:00.000000,Active,10632,5,...,United States,27.635866,26-35,29,22148.22,8,25481.85,9,0.160110,0
4,400141,200808,1,76322.83,0.0906,2021-06-04 00:00:00.000000,2024-10-23 00:00:00.000000,Active,10795,3,...,United States,38.806297,36-50,35,23123.99,11,7430.60,5,1.093213,0


## 2. Pivot Tables: Creditworthiness and Behavioral Analytics
Generating specialized tables to understand the drivers behind loan defaults.

In [2]:
# Pivot 1: Financial Health by Loan Status
pivot_financials = pd.pivot_table(risk_df, values=['Balance', 'PrincipalAmount', 'Loan_to_Balance_Ratio'], index='LoanStatusName', aggfunc='mean')
pivot_financials.to_csv('../tables/pivot_financial_health_by_loan_status.csv')
print("1. Financial Health by Loan Status:")
display(pivot_financials)

# Pivot 2: Transaction Behavior by Loan Status
pivot_behavior = pd.pivot_table(risk_df, values=['Total_Deposits', 'Total_Withdrawals', 'Total_Tx_Count'], index='LoanStatusName', aggfunc='mean')
pivot_behavior.to_csv('../tables/pivot_behavior_by_loan_status.csv')
print("\n2. Transaction Behavior by Loan Status:")
display(pivot_behavior)

# Pivot 3: Default Rate by Age Group
pivot_age = pd.pivot_table(risk_df, values='Is_Overdue', index='Age_Group', aggfunc=['count', 'mean'])
pivot_age.columns = ['Total_Loans', 'Default_Rate']
pivot_age.to_csv('../tables/pivot_default_rate_by_age.csv')
print("\n3. Default Rate by Age Group:")
display(pivot_age)

# Pivot 4: Default Rate by Customer Type
pivot_cust_type = pd.pivot_table(risk_df, values='Is_Overdue', index='TypeName_Customer', aggfunc=['count', 'mean'])
pivot_cust_type.columns = ['Total_Loans', 'Default_Rate']
pivot_cust_type.to_csv('../tables/pivot_default_rate_by_customer_type.csv')
print("\n4. Default Rate by Customer Type:")
display(pivot_cust_type)

1. Financial Health by Loan Status:


,Balance,Loan_to_Balance_Ratio,PrincipalAmount
LoanStatusName,,,
Active,49517.156996,3.234264,51963.718182
Overdue,44169.633784,3.373626,49163.180811
Paid Off,51117.039138,2.144864,52878.385517



2. Transaction Behavior by Loan Status:


,Total_Deposits,Total_Tx_Count,Total_Withdrawals
LoanStatusName,,,
Active,25668.421423,32.794466,24177.171423
Overdue,22217.422703,33.054054,23704.480541
Paid Off,26703.930862,33.620690,25051.646207



3. Default Rate by Age Group:


/var/folders/8g/_6xbb_gs2w9b7qsq4txkkhs00000gn/T/ipykernel_71605/2090451186.py:14: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_age = pd.pivot_table(risk_df, values='Is_Overdue', index='Age_Group', aggfunc=['count', 'mean'])
/var/folders/8g/_6xbb_gs2w9b7qsq4txkkhs00000gn/T/ipykernel_71605/2090451186.py:14: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_age = pd.pivot_table(risk_df, values='Is_Overdue', index='Age_Group', aggfunc=['count', 'mean'])


,Total_Loans,Default_Rate
Age_Group,,
18-25,0,NaN
26-35,80,0.10
36-50,125,0.12
50+,120,0.10



4. Default Rate by Customer Type:


,Total_Loans,Default_Rate
TypeName_Customer,,
Individual,121,0.099174
Large Enterprise,127,0.110236
Small Business,100,0.110000


## 3. Deep Statistical Analysis
Proving the statistical significance of spending behaviors on default risk.

In [3]:
# 1. Independent T-Test: Does Transaction Frequency affect Default Risk?
overdue_tx = risk_df[risk_df['LoanStatusName'] == 'Overdue']['Total_Tx_Count'].dropna()
paidoff_tx = risk_df[risk_df['LoanStatusName'] == 'Paid Off']['Total_Tx_Count'].dropna()

t_stat_tx, p_val_tx = stats.ttest_ind(overdue_tx, paidoff_tx, equal_var=False)
print(f"T-Test (Transaction Frequency) -> T-statistic: {t_stat_tx:.4f}, P-value: {p_val_tx:.4e}")

# 2. Correlation Matrix of Risk Factors
risk_factors = ['Balance', 'Age', 'Total_Tx_Count', 'Total_Deposits', 'Total_Withdrawals', 'Loan_to_Balance_Ratio', 'Is_Overdue']
corr_matrix = risk_df[risk_factors].corr()
corr_matrix.to_csv('../tables/pivot_correlation_matrix.csv')
print("\nCorrelation Matrix of Risk Factors:")
display(corr_matrix)

T-Test (Transaction Frequency) -> T-statistic: -0.4032, P-value: 6.8786e-01

Correlation Matrix of Risk Factors:


,Balance,Age,Total_Tx_Count,Total_Deposits,Total_Withdrawals,Loan_to_Balance_Ratio,Is_Overdue
Balance,1.000000,0.060851,-0.027991,-0.058239,0.052184,-0.123850,-0.058888
Age,0.060851,1.000000,0.071541,0.071427,0.027317,0.061779,0.022324
Total_Tx_Count,-0.027991,0.071541,1.000000,0.649217,0.619394,-0.038022,0.003932
Total_Deposits,-0.058239,0.071427,0.649217,1.000000,0.206542,0.001382,-0.099378
Total_Withdrawals,0.052184,0.027317,0.619394,0.206542,1.000000,0.003746,-0.018730
Loan_to_Balance_Ratio,-0.123850,0.061779,-0.038022,0.001382,0.003746,1.000000,0.003898
Is_Overdue,-0.058888,0.022324,0.003932,-0.099378,-0.018730,0.003898,1.000000


## 4. Visualizations for the Final Report
Generating specific visual insights for the lending strategy recommendations.

In [4]:
sns.set_theme(style="whitegrid")

# Graph 1: Correlation Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdYlGn', fmt=".2f", vmin=-1, vmax=1)
plt.title('Correlation Heatmap: Spending Patterns vs Default Risk')
plt.savefig('../graphs/correlation_heatmap_risk_factors.png', bbox_inches='tight')
plt.close()

# Graph 2: Loan-to-Balance Ratio Distribution by Status
plt.figure(figsize=(10, 6))
sns.kdeplot(data=risk_df, x='Loan_to_Balance_Ratio', hue='LoanStatusName', fill=True, common_norm=False, palette='Set1')
plt.title('Loan-to-Balance Ratio Distribution (Risk Assessment)')
plt.xlabel('Loan Principal / Account Balance')
plt.xlim(0, 5)
plt.savefig('../graphs/loan_to_balance_ratio_kde.png', bbox_inches='tight')
plt.close()

# Graph 3: Default Rate by Age Group
plt.figure(figsize=(8, 5))
default_rates = pivot_age.reset_index()
sns.barplot(data=default_rates, x='Age_Group', y='Default_Rate', palette='Blues_d')
plt.title('Probability of Loan Default by Age Group')
plt.ylabel('Default Rate (%)')
plt.savefig('../graphs/default_rate_by_age.png', bbox_inches='tight')
plt.close()

# Graph 4: Transaction Behavior (Deposits vs Withdrawals) by Loan Status
plt.figure(figsize=(10, 6))
behavior_melt = risk_df.melt(id_vars='LoanStatusName', value_vars=['Total_Deposits', 'Total_Withdrawals'], var_name='Transaction Type', value_name='Amount')
sns.barplot(data=behavior_melt, x='LoanStatusName', y='Amount', hue='Transaction Type', palette='coolwarm', errorbar=None)
plt.title('Average Deposit vs Withdrawal Amounts by Loan Status')
plt.savefig('../graphs/transaction_behavior_vs_loan_status.png', bbox_inches='tight')
plt.close()

print("Advanced Analytics complete! Extracted new fields (Age, Tx Frequency, Loan Ratios). Generated 5 pivot tables and 4 deep-dive graphs.")

Advanced Analytics complete! Extracted new fields (Age, Tx Frequency, Loan Ratios). Generated 5 pivot tables and 4 deep-dive graphs.


/var/folders/8g/_6xbb_gs2w9b7qsq4txkkhs00000gn/T/ipykernel_71605/1563256492.py:22: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=default_rates, x='Age_Group', y='Default_Rate', palette='Blues_d')
